# Notebook 2 of 3 — Generation (manual §5.4 step 5, Tasks 2 and 6)

**Settings:** Accelerator **GPU T4 x2**, Internet **On**.
**Input:** Add Input → Your Work → Notebooks → the saved version of **Notebook 1 (training)**.

| Generation | Eval set | Purpose |
|---|---|---|
| Qwen2.5-0.5B base | AlpacaEval 300 (seed 42) | Task 2 baseline |
| SFT best, DPO-UF best | AlpacaEval 300 (seed 42) | UltraFeedback DPO vs SFT |
| SFT best, DPO-HH best | provided HH helpful/harmless set (300) | HH-RLHF DPO vs SFT |
| SmolLM2-360M-Instruct, + LoRA adapter | persona style set (40) | LoRA persona evaluation |

Greedy decoding, identical settings for both models in each comparison. Each file is written as soon as its
model finishes; pairwise JSONL files are written before any judging (manual §5.5).

In [ ]:
# Manual section 7.1: clone the repository in Kaggle (Notebook settings: Internet ON).
import os
REPO_URL = "https://github.com/mahamasiff/cs5326-PA2-27100015.git"
REPO_DIR = "/kaggle/working/cs5326-PA2-27100015"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
import subprocess
def sh(cmd):
    """Run a shell command, show its output in this cell, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode} (see output above): {cmd}")

In [ ]:
# Manual sections 5.2 and 7.2: install dependencies without reinstalling PyTorch.
# peft/trl are added for the Task 5 LoRA run; torchao is removed to avoid the known PEFT conflict.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest "peft>=0.11.0" "trl>=0.24,<0.30"
!python -m pip install -q -e . --no-deps

In [ ]:
# Locate files from the attached input dataset (Add Input -> the saved version of the previous notebook).
import glob
def find_input(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Attach the saved notebook output that contains {pattern}"
    print(pattern, "->", hits[0])
    return hits[0]

In [ ]:
SFT = os.path.dirname(find_input("sft/best/checkpoint_info.json"))
DPO_HH = os.path.dirname(find_input("dpo_hh/best/checkpoint_info.json"))
DPO_UF = os.path.dirname(find_input("dpo_uf/best/checkpoint_info.json"))
LORA = os.path.dirname(find_input("lora_dpo/final/adapter_config.json"))
BASE, SMOL = "Qwen/Qwen2.5-0.5B", "HuggingFaceTB/SmolLM2-360M-Instruct"
ALPACA, HH, PERSONA = "eval_sets/alpaca_eval_300_seed42.json", "eval_sets/hh_rlhf_eval_300_seed42.json", "eval_sets/persona_style_eval.json"
G = "outputs/generations"
os.makedirs(G, exist_ok=True)

### Decoding settings
Greedy decoding by default (§8). If the sanity check shows repetitive loops, set a modest repetition penalty
(e.g. 1.1) here, re-run from this cell, and document the choice in the report.

In [ ]:
DECODE = "--max_new_tokens 512 --batch_size 16 --repetition_penalty 1.0"  # same for every model in a comparison

## Sanity check on 3 prompts per checkpoint (§12): path, tokenizer, prompt format, EOS, length

In [ ]:
sh(f"python scripts/generate.py --model_path {BASE} --generator qwen2.5-0.5b-base --eval_set {ALPACA} --output {G}/sanity_base.json --limit 3 {DECODE}")
sh(f"python scripts/generate.py --model_path {SFT} --generator sft --eval_set {ALPACA} --output {G}/sanity_sft.json --limit 3 {DECODE}")
sh(f"python scripts/generate.py --model_path {DPO_UF} --generator dpo_uf --eval_set {ALPACA} --output {G}/sanity_dpo_uf.json --limit 3 {DECODE}")
sh(f"python scripts/generate.py --model_path {DPO_HH} --generator dpo_hh --eval_set {HH} --output {G}/sanity_dpo_hh.json --limit 3 {DECODE}")
sh(f"python scripts/generate.py --model_path {SMOL} --adapter_path {LORA} --prompt_format chat_template --generator smollm2-lora-dpo --eval_set {PERSONA} --output {G}/sanity_lora.json --limit 3 --max_new_tokens 256")

## Task 2 — Baseline generation: Qwen2.5-0.5B base on the fixed AlpacaEval subset

In [ ]:
sh(f"python scripts/generate.py --model_path {BASE} --generator qwen2.5-0.5b-base --eval_set {ALPACA} --output {G}/alpaca_base.json {DECODE}")

## Task 6 — Evaluation generations (two models at a time, one per GPU)

In [ ]:
# UltraFeedback track: SFT vs DPO-UF on AlpacaEval (same seed-42 subset as the baseline).
sh(f"python scripts/generate.py --model_path {SFT} --generator sft --eval_set {ALPACA} --output {G}/alpaca_sft.json --device cuda:0 {DECODE} > {G}/log_alpaca_sft.txt 2>&1 & "
   f"python scripts/generate.py --model_path {DPO_UF} --generator dpo_uf --eval_set {ALPACA} --output {G}/alpaca_dpo_uf.json --device cuda:1 {DECODE} > {G}/log_alpaca_dpo_uf.txt 2>&1 & wait")

In [ ]:
# HH-RLHF track: SFT vs DPO-HH on the provided synthetic helpful/harmless set.
sh(f"python scripts/generate.py --model_path {SFT} --generator sft --eval_set {HH} --output {G}/hh_sft.json --device cuda:0 {DECODE} > {G}/log_hh_sft.txt 2>&1 & "
   f"python scripts/generate.py --model_path {DPO_HH} --generator dpo_hh --eval_set {HH} --output {G}/hh_dpo_hh.json --device cuda:1 {DECODE} > {G}/log_hh_dpo_hh.txt 2>&1 & wait")

In [ ]:
# LoRA persona track: SmolLM2-360M-Instruct with and without the LoRA-DPO adapter (its own chat template).
PERSONA_DECODE = "--max_new_tokens 256 --batch_size 16 --repetition_penalty 1.0"
sh(f"python scripts/generate.py --model_path {SMOL} --prompt_format chat_template --generator smollm2-instruct --eval_set {PERSONA} --output {G}/persona_base.json --device cuda:0 {PERSONA_DECODE} > {G}/log_persona_base.txt 2>&1 & "
   f"python scripts/generate.py --model_path {SMOL} --adapter_path {LORA} --prompt_format chat_template --generator smollm2-lora-dpo --eval_set {PERSONA} --output {G}/persona_lora.json --device cuda:1 {PERSONA_DECODE} > {G}/log_persona_lora.txt 2>&1 & wait")

In [ ]:
# Parallel runs report success even if one process failed, so verify every file is complete.
import json
expected = {"alpaca_base": ALPACA, "alpaca_sft": ALPACA, "alpaca_dpo_uf": ALPACA, "hh_sft": HH, "hh_dpo_hh": HH,
            "persona_base": PERSONA, "persona_lora": PERSONA}
for name, eval_set in expected.items():
    path = f"{G}/{name}.json"
    assert os.path.exists(path), f"{path} missing: check {G}/log_{name}.txt"
    g, n = json.load(open(path)), len(json.load(open(eval_set)))
    assert len(g) == n, f"{path} has {len(g)}/{n} outputs: re-run its cell (it resumes) and check the log"
    print(f"{name}: {len(g)} outputs, {sum(x['hit_max_new_tokens'] for x in g)} hit max_new_tokens, "
          f"mean chars {sum(len(x['output']) for x in g) / len(g):.0f}")

## Pairwise comparison files (§12): randomized A/B order, model identity kept as hidden metadata

In [ ]:
sh(f"python scripts/build_pairs.py --candidate {G}/alpaca_dpo_uf.json --candidate_name dpo_uf --baseline {G}/alpaca_sft.json --baseline_name sft --output outputs/eval/dpo_uf_vs_sft_pairs.jsonl")
sh(f"python scripts/build_pairs.py --candidate {G}/hh_dpo_hh.json --candidate_name dpo_hh --baseline {G}/hh_sft.json --baseline_name sft --output outputs/eval/dpo_hh_vs_sft_pairs.jsonl")
sh(f"python scripts/build_pairs.py --candidate {G}/persona_lora.json --candidate_name lora_dpo --baseline {G}/persona_base.json --baseline_name smollm2_instruct --output outputs/eval/lora_vs_base_pairs.jsonl")

In [ ]:
!cd outputs && zip -q -r /kaggle/working/generation_outputs.zip generations eval
!ls -lh /kaggle/working/generation_outputs.zip

**Done.** This version's outputs are the input dataset for Notebook 3 (evaluation).